![title](img/Fig1.jpg)

# Figure 1. 
Differences between FOSI and LENS simulation mean states over 1958-1978. Annual-mean precipitation (colorfill) and surface wind stress (red vectors) from (a) FOSI, (b) LENS, and (c) the difference in precipitation (LENS minus FOSI). Absolute value of PV on the 20°C isotherm surface for (d) FOSI, (e) LENS, and (f) the PV difference between (LENS minus FOSI). Zonal-mean PV (averaged between 160°E–120°W) for (g) FOSI and (h) LENS, and (i) the difference in zonal-mean PV (LENS minus FOSI). Black lines in (g)-(i) indicate climatological isotherms. Stippling in (c), (f), and (i) indicates regions where FOSI differs significantly from the LENS ensemble mean at the 95% confidence level (two-tailed t-test).

### Imports

In [ ]:
import pop_tools
import xarray as xr
import numpy as np
import cftime
import xesmf as xe
import cmocean
import matplotlib.pyplot as plt
import matplotlib.collections as mcollections
import matplotlib.patches as patches
from matplotlib.patches import Polygon
import cartopy.crs as ccrs
from scipy.signal import coherence, csd
from scipy.ndimage import gaussian_filter
from scipy.signal import find_peaks
from scipy.stats import linregress
import scipy.stats as stats
import pandas as pd
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
import xesmf as xe
import pop_tools
import cftime
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from scipy.stats import gaussian_kde

import warnings
warnings.filterwarnings('ignore')

In [ ]:
import processing_utils as proc_utils
import cesm2_lens_utils
import analysis_funcs as afuncs
import stc_funcs as stcfuncs

### Functions

In [ ]:
def regrid_SMYLE(ds, glat=1, glon=1): # from Jacob's notebook
    """
    Inputs:
        ds: xr.DataArray with coordinates that include TLAT and TLONG
    Returns:
        Regridded xr.DataArray with coordinates lat and lon
    """
    ds = ds.rename(({'ULONG': 'lon', 'ULAT': 'lat'}))
    ds_out = xe.util.grid_global(glon, glat)
    regridder = xe.Regridder(ds, ds_out, 'bilinear', periodic=True)
    regridded = regridder(ds)
    new_coords = regridded.assign_coords({'y': regridded.lat[:, 0].values, 'x': regridded.lon[0].values})
    return new_coords.drop_vars(['lat', 'lon']).rename({'x': 'lon', 'y': 'lat'})

### Set up

In [ ]:
# LENS for regridding purposes
CESMLENS_var = afuncs.LENS_for_regridding() # this is Area

#### FOSI set up
firstyear = 1959
lastyear = 2020
grid = pop_tools.get_grid('POP_gx1v7')
mask = xr.where((grid['REGION_MASK']>0) & (grid['REGION_MASK']<9), 1, np.nan)
fpath = '/glade/campaign/cesm/development/espwg/SMYLE/initial_conditions/SMYLE-FOSI/ocn/proc/tseries/month_1/'
fosi_montime_vals = [cftime.DatetimeNoLeap(1958+year, 1+month, 15) for year in range(63) for month in range(12)]

## Row 1: (a) Precipitation (colors) and surface wind stress (vectors) from LENS. (b) As in (a) but for FOSI.

### FOSI

#### Access data

In [ ]:
# Precipitation PREC_F
PREC_F = proc_utils.process_fosi_atm_var('PREC_F', fosi_montime_vals)*10e-4 # want the same units as LENS
regridder = xe.Regridder(PREC_F, CESMLENS_var[:,:,:], 'nearest_s2d', periodic=True)

regridded_PREC_F = regridder(PREC_F).sel(time=slice('1958-01', '2020-12'))
regridded_PREC_F['time'] = CESMLENS_var.sel(time=slice('1958-01', '2020-12'))['time']

landmask = xr.where(regridded_PREC_F > 0., 1., 0.)

In [ ]:
field = 'TAUX'
fname = f'g.e22.GOMIPECOIAF_JRA-1p4-2018.TL319_g17.SMYLE.005.pop.h.{field}.030601-036812.nc'
fpath = '/glade/campaign/cesm/development/espwg/SMYLE/initial_conditions/SMYLE-FOSI/ocn/proc/tseries/month_1/'
ds_smyle_fosi_var = xr.open_dataset(fpath+fname)[field]
ds_smyle_fosi_var['time'] = fosi_montime_vals
var_fosi = ds_smyle_fosi_var.isel(time=slice(0, 240)).compute()
fosi_1deg_wzeros_var = regrid_SMYLE(var_fosi)
fosi_1deg_var_TAUX = fosi_1deg_wzeros_var.where(fosi_1deg_wzeros_var!=0, np.nan)

field = 'TAUY'
fname = f'g.e22.GOMIPECOIAF_JRA-1p4-2018.TL319_g17.SMYLE.005.pop.h.{field}.030601-036812.nc'
fpath = '/glade/campaign/cesm/development/espwg/SMYLE/initial_conditions/SMYLE-FOSI/ocn/proc/tseries/month_1/'
ds_smyle_fosi_var = xr.open_dataset(fpath+fname)[field]
ds_smyle_fosi_var['time'] = fosi_montime_vals
var_fosi = ds_smyle_fosi_var.isel(time=slice(0, 240)).compute()
fosi_1deg_wzeros_var = regrid_SMYLE(var_fosi)
fosi_1deg_var_TAUY = fosi_1deg_wzeros_var.where(fosi_1deg_wzeros_var!=0, np.nan)

## Regridding
regridder = xe.Regridder(fosi_1deg_var_TAUY, CESMLENS_var[:,:,:], 'nearest_s2d', periodic=True)
regridded_fosi_TAUX = regridder(fosi_1deg_var_TAUX)/10
regridded_fosi_TAUY = regridder(fosi_1deg_var_TAUY)/10

FOSI_TAUX = regridded_fosi_TAUX.mean(dim='time')
FOSI_TAUY = regridded_fosi_TAUY.mean(dim='time')

# # Calculate seasonal means for DJF (boreal winter)
# FOSI_TAUX_DJF = regridded_fosi_TAUX.where(regridded_fosi_TAUX.time.dt.month.isin([12, 1, 2]), drop=True).mean(dim='time')
# FOSI_TAUY_DJF = regridded_fosi_TAUY.where(regridded_fosi_TAUY.time.dt.month.isin([12, 1, 2]), drop=True).mean(dim='time')

#### Plotting

In [ ]:
fig, ax = plt.subplots(figsize=(4, 3),
                      subplot_kw={'projection': ccrs.PlateCarree(central_longitude=180, globe=None)})

####### STATICS
ax.add_feature(cfeature.LAND, color='lightgray', zorder=100)
ax.add_feature(cfeature.COASTLINE, linewidth=1., zorder=100)
ax.grid(c='k', linestyle='dashed', alpha=0.2, zorder=4)
ax.set_extent([120, 284, -30, 30], crs=ccrs.PlateCarree())
gl = ax.gridlines(draw_labels={'left': True, 'bottom': True, 'right': False, 'top': False}, 
                  zorder=4, linestyle='--', alpha=0.0)
gl.xlabel_style = {'size': 10, 'color':'k'}  # Longitude labels
gl.ylabel_style = {'size': 10, 'color':'k'}  # Latitude labels
ax.axhline(y=0, color='k', linestyle='-', linewidth=1, zorder=5)
ax.spines['geo'].set_edgecolor('black')
ax.spines['geo'].set_linewidth(1.5)
ax.set_aspect('auto')

FOSI_PRECT = regridded_PREC_F.isel(time=slice(0, 240)).mean(dim='time').sel(
        lat=slice(-60, 60), 
        lon=slice(120, 295))

# #### CONTOURF AND CONTOUR
contourf_plot = FOSI_PRECT.plot.contourf(
    x='lon', y='lat', cmap='Blues', add_colorbar=False, transform=ccrs.PlateCarree(),
    vmin=0, vmax=1.0e-7, levels=11, alpha=1)

step = 5
step_lon = step
step_lat = step

lon_sub = FOSI_TAUX.lon.values[::5]
lat_sub = FOSI_TAUX.lat.values[::4]
TAUX_sub = FOSI_TAUX.sel(lon=lon_sub, lat=lat_sub).values
TAUY_sub = FOSI_TAUY.sel(lon=lon_sub, lat=lat_sub).values

# Create meshgrid for quiver
lon_grid, lat_grid = np.meshgrid(lon_sub, lat_sub)

# Calculate a good scale automatically
vector_magnitudes = np.sqrt(TAUX_sub**2 + TAUY_sub**2)
median_mag = np.median(vector_magnitudes[~np.isnan(vector_magnitudes)])
scale_value = median_mag * 30  # Adjust multiplier: smaller = bigger arrows

Q = ax.quiver(lon_grid, lat_grid, TAUX_sub, TAUY_sub,
              transform=ccrs.PlateCarree(),
              scale=scale_value,  # Dynamic scaling
              width=0.004,  # Slightly thicker
              headwidth=3,  # Slightly larger head
              headlength=3,
              headaxislength=5,
              color='red',
              edgecolor=None, 
              linewidth=0.1,
              alpha=0.9,  # Slight transparency
              zorder=10)

ax.quiverkey(Q, 0.78, -0.10, 0.1, 
             '0.1 N/m²', 
             labelpos='E',
             coordinates='axes',
             fontproperties={'size': 10, 'weight': 'normal'})

plt.title('(a) FOSI: Precipitation', fontsize=11, fontweight='bold', zorder=12, loc='left')
plt.tight_layout()
plt.show()

### LENS

#### Access data

In [ ]:
ENS_MEMBERS_LIST = [0, 65,  32, 85, 61, 90, 80, 68, 73, 49] 

LENS_PRECT_list = []
for ENS_MEMB in ENS_MEMBERS_LIST:
    PRECT = afuncs.atm_var_ens(ENS_MEMB, 'PRECT')
    PRECT_noland = PRECT.sel(
        lat=slice(-60, 60), 
        lon=slice(120, 295))*landmask

    PRECT_noland_ds = xr.where(PRECT_noland > 0., PRECT_noland, np.nan)
    PRECT_noland_ds_mean = PRECT_noland_ds.isel(time=slice(0, 240)).mean(dim='time')
    LENS_PRECT_list.append(PRECT_noland_ds_mean)

LENS_PRECT = xr.concat(LENS_PRECT_list, dim='ensemble')

In [ ]:
ENS_MEMBERS_LIST = [0, 65,  32, 85, 61, 90, 80, 68, 73, 49] 

LENS_TAUX_list = []
LENS_TAUY_list = []

for ENS_MEMB in ENS_MEMBERS_LIST:
    TAUX = afuncs.atm_var_ens(ENS_MEMB, 'TAUX')
    TAUY = afuncs.atm_var_ens(ENS_MEMB, 'TAUY')
    TAUX_noland = TAUX.sel(
        lat=slice(-60, 60), 
        lon=slice(120, 295)).isel(time=slice(0, 240))
    TAUY_noland = TAUY.sel(
        lat=slice(-60, 60), 
        lon=slice(120, 295)).isel(time=slice(0, 240))

    TAUX_noland_ds = TAUX_noland
    TAUY_noland_ds = TAUY_noland
    
    LENS_TAUX_list.append(TAUX_noland_ds)
    LENS_TAUY_list.append(TAUY_noland_ds)

LENS_TAUX = xr.concat(LENS_TAUX_list, dim='ensemble')
LENS_TAUY = xr.concat(LENS_TAUY_list, dim='ensemble')

# Calculate seasonal means for DJF (boreal winter)
LENS_TAUX_DJF = LENS_TAUX.where(LENS_TAUX.time.dt.month.isin([12, 1, 2]), drop=True).mean(dim='time')
LENS_TAUY_DJF = LENS_TAUY.where(LENS_TAUY.time.dt.month.isin([12, 1, 2]), drop=True).mean(dim='time')

# Then use these seasonal datasets in your plot:
TAUX_mean = LENS_TAUX_DJF.mean(dim='ensemble')
TAUY_mean = LENS_TAUY_DJF.mean(dim='ensemble')

LENS_TAUX_DJF_mean = TAUX_mean*-1
LENS_TAUY_DJF_mean = TAUY_mean*-1

LENS_TAUX_mean = LENS_TAUX.mean(dim='time').mean(dim='ensemble')*-1
LENS_TAUY_mean = LENS_TAUY.mean(dim='time').mean(dim='ensemble')*-1

#### Plotting

In [ ]:
fig, ax = plt.subplots(figsize=(4, 3),
                      subplot_kw={'projection': ccrs.PlateCarree(central_longitude=180, globe=None)})

####### STATICS
ax.add_feature(cfeature.LAND, color='lightgray', zorder=100)
ax.add_feature(cfeature.COASTLINE, linewidth=1., zorder=100)
ax.grid(c='k', linestyle='dashed', alpha=0.2, zorder=4)
ax.set_extent([120, 284, -30, 30], crs=ccrs.PlateCarree())
gl = ax.gridlines(draw_labels={'left': True, 'bottom': True, 'right': False, 'top': False}, 
                  zorder=4, linestyle='--', alpha=0.0)
gl.xlabel_style = {'size': 10, 'color':'k'}  # Longitude labels
gl.ylabel_style = {'size': 10, 'color':'k'}  # Latitude labels
ax.axhline(y=0, color='k', linestyle='-', linewidth=1, zorder=5)
ax.spines['geo'].set_edgecolor('black')
ax.spines['geo'].set_linewidth(1.5)
ax.set_aspect('auto')

contourf_plot = LENS_PRECT.mean(dim='ensemble').plot.contourf(
    x='lon', y='lat', cmap='Blues', add_colorbar=False, transform=ccrs.PlateCarree(),
    vmin=0, vmax=1.0e-7, levels=11, alpha=1)

step_lon = step
step_lat = step

lon_sub = LENS_TAUX_mean.lon.values[::5]
lat_sub = LENS_TAUX_mean.lat.values[::4]
TAUX_sub = LENS_TAUX_mean.sel(lon=lon_sub, lat=lat_sub).values
TAUY_sub = LENS_TAUY_mean.sel(lon=lon_sub, lat=lat_sub).values

# Create meshgrid for quiver
lon_grid, lat_grid = np.meshgrid(lon_sub, lat_sub)

# Calculate a good scale automatically
vector_magnitudes = np.sqrt(TAUX_sub**2 + TAUY_sub**2)
median_mag = np.median(vector_magnitudes[~np.isnan(vector_magnitudes)])
scale_value = median_mag * 30  # Adjust multiplier: smaller = bigger arrows

Q = ax.quiver(lon_grid, lat_grid, TAUX_sub, TAUY_sub,
              transform=ccrs.PlateCarree(),
              scale=scale_value,
              width=0.004,
              headwidth=3,
              headlength=3,
              headaxislength=5,
              color='red',
              edgecolor=None, 
              linewidth=0.1,
              alpha=0.9,
              zorder=10)

ax.quiverkey(Q, 0.78, -0.10, 0.1, 
             '0.1 N/m²', 
             labelpos='E',
             coordinates='axes',
             fontproperties={'size': 10, 'weight': 'normal'})

plt.title('(b) LENS: Precipitation', fontsize=11, fontweight='bold', zorder=12, loc='left')
plt.tight_layout()
plt.show()

### LENS minus FOSI

In [ ]:
diff_PRECT = LENS_PRECT.mean(dim='ensemble') - FOSI_PRECT

#### Plotting

In [ ]:
# --- Calculate significance ---
lens_mean = LENS_PRECT.mean(dim='ensemble')
lens_std  = LENS_PRECT.std(dim='ensemble')
n = 10

t_stat = (FOSI_PRECT - lens_mean) / (lens_std / np.sqrt(n))
p_value = stats.t.sf(np.abs(t_stat), df=n-1) * 2  # two-tailed

sig_mask = p_value < 0.05  # True where significant

# --- Plot ---
fig, ax = plt.subplots(figsize=(4, 3),
                       subplot_kw={'projection': ccrs.PlateCarree(central_longitude=180)})
ax.add_feature(cfeature.LAND, color='lightgray', zorder=100)
ax.add_feature(cfeature.COASTLINE, linewidth=1., zorder=100)
ax.set_extent([120, 284, -30, 30], crs=ccrs.PlateCarree())
gl = ax.gridlines(draw_labels={'left': True, 'bottom': True, 'right': False, 'top': False},
                  zorder=4, linestyle='--', alpha=0.0)
gl.xlabel_style = {'size': 10, 'color': 'k'}
gl.ylabel_style = {'size': 10, 'color': 'k'}
ax.axhline(y=0, color='k', linestyle='-', linewidth=1, zorder=5)
ax.spines['geo'].set_edgecolor('black')
ax.spines['geo'].set_linewidth(1.5)
ax.set_aspect('auto')

# Main difference plot
contourf_plot = diff_PRECT.plot.contourf(
    x='lon', y='lat', cmap='PRGn', add_colorbar=False, transform=ccrs.PlateCarree(),
    vmin=-3.0e-8, vmax=3.0e-8, levels=13, alpha=1)

# Stippling where significant
# Subsample to avoid overcrowding (every 3rd point)
step = 2
lons = diff_PRECT.lon.values[::step]
lats = diff_PRECT.lat.values[::step]
mask = sig_mask[::step, ::step]

lon2d, lat2d = np.meshgrid(lons, lats)
ax.scatter(lon2d[mask], lat2d[mask],
           s=0.3, color='k', alpha=0.6,
           transform=ccrs.PlateCarree(), zorder=101)

plt.title('(c) LENS - FOSI: Precipitation', fontsize=11, fontweight='bold', zorder=12, loc='left')
plt.tight_layout()
plt.show()

### Colorbars

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(1.8, 3))

# Blues colorbar
density_sm1 = plt.cm.ScalarMappable(cmap='Blues', norm=plt.Normalize(0, 1e-7))
cbar1 = plt.colorbar(density_sm1, cax=ax1, orientation='vertical')
cbar1.set_label('Precipitation (×10⁻⁸ m/s)', fontsize=10, labelpad=10)
cbar1.set_ticks(np.arange(0, 11, 2) * 1e-8)
cbar1.set_ticklabels(['0', '2', '4', '6', '8', '10'])
cbar1.ax.tick_params(labelsize=10)

# PRGn colorbar  
density_sm2 = plt.cm.ScalarMappable(cmap='PRGn', norm=plt.Normalize(-3e-8, 3e-8))
cbar2 = plt.colorbar(density_sm2, cax=ax2, orientation='vertical')
cbar2.set_label('ΔPrecipitation (×10⁻⁸ m/s)', fontsize=10, labelpad=10)
cbar2.set_ticks(np.arange(-3, 4, 1) * 1e-8)
cbar2.set_ticklabels(['-3', '-2', '-1', '0', '1', '2', '3'])
cbar2.ax.tick_params(labelsize=10)
plt.tight_layout()
plt.show()

## Row 2: Magnitude of PV (|PV|) on the 20°C isotherm surface for (c) LENS ensemble mean, (d) FOSI, and (e) their difference (LENS minus FOSI), where red indicates larger |PV| in LENS.

In [ ]:
time_start = 0; time_end = 240

### FOSI

#### Access data

In [ ]:
FOSI_PV = xr.open_dataset('/glade/derecho/scratch/cassiacai/regridded_PV_FOSI.nc')
FOSI_PV_mean = FOSI_PV.PV.isel(time=slice(time_start,time_end)).isel(TEMP = 2).mean(dim='time').compute()
FOSI_PV_1deg = proc_utils.regrid_SMYLE(FOSI_PV_mean)
regridded_FOSI_PV = np.absolute(regridder(FOSI_PV_1deg))

#### Plotting

In [ ]:
fig, ax = plt.subplots(figsize=(4, 3),
                      subplot_kw={'projection': ccrs.PlateCarree(central_longitude=180, globe=None)})

ax.add_feature(cfeature.LAND, color='lightgray', zorder=100)
ax.add_feature(cfeature.COASTLINE, linewidth=1., zorder=100)
ax.set_extent([120, 284, -30, 30], crs=ccrs.PlateCarree())
gl = ax.gridlines(draw_labels={'left': True, 'bottom': True, 'right': False, 'top': False}, 
                  zorder=4, linestyle='--', alpha=0.0)
gl.xlabel_style = {'size': 10, 'color':'k'}  # Longitude labels
gl.ylabel_style = {'size': 10, 'color':'k'}  # Latitude labels
ax.axhline(y=0, color='k', linestyle='-', linewidth=1, zorder=5)
ax.spines['geo'].set_edgecolor('black')
ax.spines['geo'].set_linewidth(1.5)
ax.set_aspect('auto')

contourf_plot = regridded_FOSI_PV.sel(lat=slice(-30, 30), lon=slice(120,290)).plot.contourf(
    vmin=0, vmax=2e-11, levels=11,
    cmap='OrRd', transform=ccrs.PlateCarree(), add_colorbar=False)

plt.title('')
plt.title('(d) FOSI: Potential Vorticity', fontsize=11, fontweight='bold', zorder=12, loc='left')
plt.tight_layout()
plt.show()

### LENS

#### Access data

In [ ]:
ENS_MEMBERS_LIST = [0, 65,  32, 85, 61, 90, 80, 68, 73, 49] 

file_paths = [f'/glade/derecho/scratch/cassiacai/regridded_PV_{member}.nc' for member in ENS_MEMBERS_LIST]
LENS_PV_all = xr.open_mfdataset(file_paths, combine='nested', concat_dim='ensemble')
LENS_PV_mean = LENS_PV_all.PV.isel(time=slice(time_start,time_end)).isel(TEMP = 2).mean(dim='time').compute()
LENS_PV_1deg = proc_utils.regrid_SMYLE(LENS_PV_mean)

regridder = xe.Regridder(LENS_PV_1deg, CESMLENS_var[:,:], 'nearest_s2d', periodic=True)

regridded_LENS_PV = np.absolute(regridder(LENS_PV_1deg))

#### Plotting

In [ ]:
fig, ax = plt.subplots(figsize=(4, 3),
                      subplot_kw={'projection': ccrs.PlateCarree(central_longitude=180, globe=None)})

ax.add_feature(cfeature.LAND, color='lightgray', zorder=100)
ax.add_feature(cfeature.COASTLINE, linewidth=1., zorder=100)
ax.set_extent([120, 284, -30, 30], crs=ccrs.PlateCarree())
gl = ax.gridlines(draw_labels={'left': True, 'bottom': True, 'right': False, 'top': False}, 
                  zorder=4, linestyle='--', alpha=0.0)
gl.xlabel_style = {'size': 10, 'color':'k'}  # Longitude labels
gl.ylabel_style = {'size': 10, 'color':'k'}  # Latitude labels
ax.axhline(y=0, color='k', linestyle='-', linewidth=1, zorder=5)
ax.spines['geo'].set_edgecolor('black')
ax.spines['geo'].set_linewidth(1.5)
ax.set_aspect('auto')

contourf_plot = regridded_LENS_PV.mean(dim='ensemble').sel(lat=slice(-30, 30), lon=slice(120,290)).plot.contourf(
    vmin=0, vmax=2e-11, levels=11,
    cmap='OrRd', transform=ccrs.PlateCarree(), add_colorbar=False)

plt.title('')
plt.title('(e) LENS: Potential Vorticity', fontsize=11, fontweight='bold', zorder=12, loc='left')
plt.tight_layout()
plt.show()

### LENS minus FOSI

In [ ]:
diff_PV = regridded_LENS_PV.mean(dim='ensemble') - regridded_FOSI_PV

#### Plotting

In [ ]:
# --- Calculate significance ---
lens_mean = regridded_LENS_PV.mean(dim='ensemble')
lens_std  = regridded_LENS_PV.std(dim='ensemble')
n = 10

t_stat = (regridded_FOSI_PV - lens_mean) / (lens_std / np.sqrt(n))
p_value = stats.t.sf(np.abs(t_stat), df=n-1) * 2  # two-tailed

sig_mask = p_value < 0.05  # True where significant

# --- Plot ---
fig, ax = plt.subplots(figsize=(4, 3),
                       subplot_kw={'projection': ccrs.PlateCarree(central_longitude=180)})
ax.add_feature(cfeature.LAND, color='lightgray', zorder=100)
ax.add_feature(cfeature.COASTLINE, linewidth=1., zorder=100)
ax.set_extent([120, 284, -30, 30], crs=ccrs.PlateCarree())
gl = ax.gridlines(draw_labels={'left': True, 'bottom': True, 'right': False, 'top': False},
                  zorder=4, linestyle='--', alpha=0.0)
gl.xlabel_style = {'size': 10, 'color': 'k'}
gl.ylabel_style = {'size': 10, 'color': 'k'}
ax.axhline(y=0, color='k', linestyle='-', linewidth=1, zorder=5)
ax.spines['geo'].set_edgecolor('black')
ax.spines['geo'].set_linewidth(1.5)
ax.set_aspect('auto')
contourf_plot = diff_PV.sel(lat=slice(-40, 40), lon=slice(120,290)).plot.contourf(
    vmin=-0.3e-11, vmax=0.3e-11, levels=16,
    cmap=cmocean.cm.balance,transform=ccrs.PlateCarree(), add_colorbar=False)

# Stippling where significant
# Subsample to avoid overcrowding (every 3rd point)
step = 2
lons = diff_PV.lon.values[::step]
lats = diff_PV.lat.values[::step]
mask = sig_mask[::step, ::step]
plt.title('')

lon2d, lat2d = np.meshgrid(lons, lats)
ax.scatter(lon2d[mask], lat2d[mask],
           s=0.3, color='k', alpha=0.6,
           transform=ccrs.PlateCarree(), zorder=101)

plt.title('(f) LENS - FOSI: Precipitation', fontsize=11, fontweight='bold', zorder=12, loc='left')
plt.tight_layout()
plt.show()

### Colorbar

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(1.7, 3))

# Blues colorbar
density_sm1 = plt.cm.ScalarMappable(cmap='OrRd', norm=plt.Normalize(0, 2e-11))
cbar1 = plt.colorbar(density_sm1, cax=ax1, orientation='vertical')
cbar1.set_label('PV (×10⁻¹¹ s⁻¹ cm⁻¹)', fontsize=10)
tick_values1 = np.arange(0, 2.1, 0.5)  # 0, 0.5, 1.0, 1.5, 2.0
cbar1.set_ticks(tick_values1 * 1e-11)
cbar1.set_ticklabels([f'{x:.1f}' for x in tick_values1])
cbar1.ax.tick_params(labelsize=10)

# Balance colorbar  
density_sm2 = plt.cm.ScalarMappable(cmap=cmocean.cm.balance, norm=plt.Normalize(-0.3e-11, 0.3e-11))
cbar2 = plt.colorbar(density_sm2, cax=ax2, orientation='vertical')
cbar2.set_label('ΔPV (×10⁻¹² s⁻¹ cm⁻¹)', fontsize=10)
tick_values2 = np.arange(-3, 3.1, 1)  # -0.3, -0.2, -0.1, 0, 0.1, 0.2, 0.3
cbar2.set_ticks(tick_values2 * 1e-12)
cbar2.set_ticklabels([f'{x:.0f}' for x in tick_values2])
cbar2.ax.tick_params(labelsize=10)

plt.tight_layout()
plt.show()

## Row 3: Zonal-mean |PV| profile (averaged XX°E–XX°W) for (f) LENS and (g) FOSI, highlighting the vertical structure of the subtropical PV maxima

### FOSI

#### Access data

In [ ]:
field = 'PV'
fpath = '/glade/campaign/cesm/development/espwg/SMYLE/initial_conditions/SMYLE-FOSI/ocn/proc/tseries/month_1/'
fname = f'g.e22.GOMIPECOIAF_JRA-1p4-2018.TL319_g17.SMYLE.005.pop.h.{field}.030601-036812.nc'
ds_smyle_fosi_PV = xr.open_dataset(fpath+fname)[field].isel(z_t = slice(0,27))[:,:, 72:303,140:293].compute()
ds_smyle_fosi_PV['time'] = fosi_montime_vals

In [ ]:
field = 'TEMP'
fpath = '/glade/campaign/cesm/development/espwg/SMYLE/initial_conditions/SMYLE-FOSI/ocn/proc/tseries/month_1/'
fname = f'g.e22.GOMIPECOIAF_JRA-1p4-2018.TL319_g17.SMYLE.005.pop.h.{field}.030601-036812.nc'
ds_smyle_fosi_TEMP = xr.open_dataset(fpath+fname)[field].isel(z_t = slice(0,27))[:,:, 72:303,140:293].compute()
ds_smyle_fosi_TEMP['time'] = fosi_montime_vals

In [ ]:
FOSI_PV_mean = ds_smyle_fosi_PV.isel(time=slice(0, 240)).mean(dim='time')
FOSI_TEMP_mean = ds_smyle_fosi_TEMP.isel(time=slice(0, 240)).mean(dim='time')

#### Plotting

In [ ]:
ds_smyle_fosi_PV.isel(time=slice(0, 240)).mean(dim='time')

plt.figure(figsize=(4,3))
plt.contourf(
    FOSI_PV_mean.ULAT.mean(dim='nlon'), 
    FOSI_PV_mean.z_t/100, 
    np.abs(FOSI_PV_mean.isel(nlon=slice(37, 109)).mean(dim='nlon')), cmap='OrRd',levels=np.linspace(0, 1.3e-11, 14))

cs = plt.contour(
    FOSI_TEMP_mean.ULAT.mean(dim='nlon'), 
    FOSI_TEMP_mean.z_t/100, 
    FOSI_TEMP_mean.isel(nlon=slice(37, 109)).mean(dim='nlon'), 
    levels=np.asarray([16, 18, 20, 22, 24, 26, 28]),
    colors='k',
    linewidths=0.8
)

# Add contour labels
plt.clabel(cs, inline=True, fontsize=10, fmt='%.0f')

plt.grid(c='k', alpha=0.1, linestyle='dashed')
plt.axvline(x=0, c='k', linestyle='dashed', linewidth=0.5)
plt.ylim(250, 5)
plt.xlim(-30, 30)
plt.xticks(fontsize=10)
plt.yticks(fontsize=10)
plt.xlabel('Latitude (°)', fontsize=10)
plt.ylabel('Depth (m)', fontsize=10)
plt.title('(g) FOSI: Potential Vorticity', fontsize=11, fontweight='bold', zorder=12, loc='left')
plt.tight_layout()
plt.show()

### LENS

#### Access data

In [ ]:
ENS_MEMBERS_LIST = [0, 65,  32, 85, 61, 90, 80, 68, 73, 49] 

LENS_PV_list = []
for ENS_MEMB in ENS_MEMBERS_LIST:
    print(ENS_MEMB)
    LENS_PV = afuncs.ocn_var_ens(ENS_MEMB, 'PV').isel(z_t=slice(0, 27))[:,:, 72:303,140:293].isel(time=slice(0, 240)).compute()
    
    LENS_PV_list.append(LENS_PV)

LENS_PV_combine = xr.concat(LENS_PV_list, dim='ensemble')

In [ ]:
LENS_TEMP_list = []
for ENS_MEMB in ENS_MEMBERS_LIST:
    print(ENS_MEMB)
    LENS_TEMP = afuncs.ocn_var_ens(ENS_MEMB, 'TEMP').isel(z_t=slice(0, 27))[:,:, 72:303,140:293].isel(time=slice(0, 240)).compute()
    
    LENS_TEMP_list.append(LENS_TEMP)

LENS_TEMP_combine = xr.concat(LENS_TEMP_list, dim='ensemble')

In [ ]:
LENS_PV_ens_mean = LENS_PV_combine.mean(dim='ensemble')
LENS_TEMP_ens_mean = LENS_TEMP_combine.mean(dim='ensemble')

#### Plotting

In [ ]:
plt.figure(figsize=(4,3))

# Filled contour for PV
cf = plt.contourf(
    LENS_PV_ens_mean.mean(dim='time').ULAT.mean(dim='nlon'), 
    LENS_PV_ens_mean.mean(dim='time').z_t/100, 
    np.abs(LENS_PV_ens_mean.mean(dim='time').isel(nlon=slice(37, 109)).mean(dim='nlon')), 
    cmap='OrRd',
    levels=np.linspace(0, 1.3e-11, 14)
)

cs = plt.contour(
    LENS_PV_ens_mean.mean(dim='time').ULAT.mean(dim='nlon'), 
    LENS_PV_ens_mean.mean(dim='time').z_t/100, 
    LENS_TEMP_ens_mean.mean(dim='time').isel(nlon=slice(37, 109)).mean(dim='nlon'), 
    levels=np.asarray([16, 18, 20, 22, 24, 26, 28]),
    colors='k',
    linewidths=0.8
)

# Add contour labels
plt.clabel(cs, inline=True, fontsize=10, fmt='%.0f')

plt.grid(c='k', alpha=0.1, linestyle='dashed')
plt.axvline(x=0, c='k', linestyle='dashed', linewidth=0.5)
plt.ylim(250, 5)
plt.xlim(-30, 30)
plt.xticks(fontsize=10)
plt.yticks(fontsize=10)
plt.xlabel('Latitude (°)', fontsize=10)
plt.ylabel('Depth (m)', fontsize=10)
plt.title('(h) LENS: Potential Vorticity', fontsize=11, fontweight='bold', zorder=12, loc='left')
plt.tight_layout()
plt.show()

### LENS - FOSI

In [ ]:
LENS_PV_ens_slice = LENS_PV_ens.isel(nlon=slice(37, 109)).mean(dim='nlon').mean(dim='time')
FOSI_PV_slice = FOSI_PV_mean.isel(nlon=slice(37, 109)).mean(dim='nlon')

In [ ]:
# --- Calculate significance ---
lens_mean = LENS_PV_ens_slice.mean(dim='ensemble')
lens_std  = LENS_PV_ens_slice.std(dim='ensemble')
n = 10

t_stat = (FOSI_PV_slice - lens_mean) / (lens_std / np.sqrt(n))
p_value = stats.t.sf(np.abs(t_stat), df=n-1) * 2  # two-tailed

sig_mask = p_value < 0.05  # True where significant

In [ ]:
plt.figure(figsize=(4, 3))
plt.contourf(
    FOSI_PV_mean.ULAT.mean(dim='nlon'),
    FOSI_PV_mean.z_t/100,
    diff_PV_zonal,
    cmap=cmocean.cm.balance, levels=np.linspace(-0.5e-11, 0.5e-11, 21))

# Stippling
lats = FOSI_PV_mean.ULAT.mean(dim='nlon').values
deps = FOSI_PV_mean.z_t.values / 100
lat2d, dep2d = np.meshgrid(lats, deps)

plt.scatter(lat2d[sig_mask], dep2d[sig_mask],
            s=0.3, color='k', alpha=0.6, zorder=5)

plt.grid(c='k', alpha=0.1, linestyle='dashed')
plt.axvline(x=0, c='k', linestyle='dashed', linewidth=0.5)
plt.ylim(250, 5)
plt.xlim(-30, 30)
plt.xticks(fontsize=10)
plt.yticks(fontsize=10)
plt.xlabel('Latitude (°)', fontsize=10)
plt.ylabel('Depth (m)', fontsize=10)
plt.title('(i) LENS - FOSI: Potential Vorticity', fontsize=11, fontweight='bold', zorder=12, loc='left')
plt.tight_layout()
plt.show()

### Colorbar

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(1.7, 3))

# Blues colorbar
density_sm1 = plt.cm.ScalarMappable(cmap='OrRd', norm=plt.Normalize(0, 13e-12))
cbar1 = plt.colorbar(density_sm1, cax=ax1, orientation='vertical')
cbar1.set_label('PV (×10⁻¹² s⁻¹ cm⁻¹)', fontsize=10)
tick_values1 = np.arange(0, 14, 2)  # 0, 0.5, 1.0, 1.5, 2.0
cbar1.set_ticks(tick_values1 * 1e-12)
cbar1.set_ticklabels([f'{x:.0f}' for x in tick_values1])
cbar1.ax.tick_params(labelsize=10)

# Balance colorbar  
density_sm2 = plt.cm.ScalarMappable(cmap=cmocean.cm.balance, norm=plt.Normalize(-5e-12, 5e-12))
cbar2 = plt.colorbar(density_sm2, cax=ax2, orientation='vertical')
cbar2.set_label('ΔPV (×10⁻¹² s⁻¹ cm⁻¹)', fontsize=10)
tick_values2 = np.arange(-5, 5.1, 1)  # -0.3, -0.2, -0.1, 0, 0.1, 0.2, 0.3
cbar2.set_ticks(tick_values2 * 1e-12)
cbar2.set_ticklabels([f'{x:.0f}' for x in tick_values2])
cbar2.ax.tick_params(labelsize=10)

plt.tight_layout()
plt.show()